# Session 8: Implementation in Python — inverse problems, loss balancing, and 2D Poisson equation

We build on the forward Burgers' equation from [Session 7](Session7.ipynb). This session tackles three topics:
1. **Inverse problems** — inferring unknown PDE parameters from noisy data.
2. **Loss weighting strategies** — balancing competing loss terms during training.
3. **2D Poisson equation** — extending PINNs to a 2D spatial domain.

Inverse problems are where physics-informed neural networks (PINNs) truly excel in real-world physics: identifying material properties from experiments, inferring viscosity from flow measurements, or estimating wave speeds from seismic data.

## 1. Inverse problems in PINNs

In a **forward** problem, all partial differential equation (PDE) parameters are known and we seek the solution field $u$.

In an **inverse** problem, some parameters are unknown and must be inferred alongside $u$ from sparse, noisy observations.

The key insight: make the unknown parameter a **learnable `nn.Parameter`** in the model. PyTorch's optimiser then updates it jointly with the network weights during training.

The loss becomes:
$$
\mathcal{L} = \mathcal{L}_{\text{PDE}}(\theta, \nu) + \lambda_{\text{IC}} \mathcal{L}_{\text{IC}} + \lambda_{\text{data}} \mathcal{L}_{\text{data}}
$$

where $\mathcal{L}_{\text{data}}$ penalises the discrepancy between the network and the (noisy) measurements.

Illustrated below is a general inverse PINN setup:

![PINN inverse setup](figures/fig8_1.png)
![Parameter identification](figures/fig8_2.png)
![Noisy data scenario](figures/fig8_3.jpg)
![Inferred field](figures/fig8_4.webp)
![Convergence of inferred parameter](figures/fig8_5.png)
![Error landscape](figures/fig8_6.png)

## 2. Hands-on: inverse Burgers' equation with noisy data

We modify the [Session 7](Session7.ipynb) Burgers' PINN:
- Add sparse noisy measurements of $u$.
- Make the viscosity $\nu$ a learnable parameter.
- Report the inferred $\nu$ after training.

### 2.1 Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
class InversePINN(nn.Module):
    """Burgers' PINN with learnable viscosity nu."""
    def __init__(self, layers=[2, 64, 64, 64, 64, 1]):
        super().__init__()
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i+1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)
        # Initialise at 2× the true value (0.02/π vs true 0.01/π) so the
        # network must correct downward — a deliberate test of convergence.
        self.log_nu = nn.Parameter(torch.log(torch.tensor(0.02 / np.pi)))

    def forward(self, X):
        return self.net(X)

    @property
    def nu(self):
        return torch.exp(self.log_nu)   # keep nu positive via log-parameterisation

### 2.2 Generate synthetic noisy observations

In a real scenario these would be experimental measurements. Here we simulate them using the low-amplitude approximation $u \approx -\sin(\pi x)\, e^{-\nu \pi^2 t}$, which holds for the linearised equation, and add Gaussian noise.

> **Note on the approximation.** The true Burgers' solution for $u(x,0) = -\sin(\pi x)$ does not have a simple closed form. We use the linearised approximation (valid when the amplitude remains small) purely for convenience in generating synthetic data. In a real inverse problem you would use measurements from experiment or a full numerical solve. The PINN's PDE loss still enforces the exact nonlinear equation — only the synthetic observations come from the approximate formula.

In [ ]:
true_nu = 0.01 / np.pi
N_data  = 1000
torch.manual_seed(0)

data_x = 2 * torch.rand(N_data, 1) - 1
data_t = torch.rand(N_data, 1)
data_xt = torch.cat([data_x, data_t], dim=1).to(device)

u_true_data = -torch.sin(np.pi * data_x) * torch.exp(-true_nu * np.pi**2 * data_t)
noise_std = 0.02
u_noisy = (u_true_data + noise_std * torch.randn_like(u_true_data)).to(device)

### 2.3 Sample collocation and initial condition (IC) points

In [ ]:
N_colloc = 20000
N_ic     = 500
N_bc     = 200

colloc_x = 2 * torch.rand(N_colloc, 1, device=device) - 1
colloc_t = torch.rand(N_colloc, 1, device=device)
colloc = torch.cat([colloc_x, colloc_t], dim=1).requires_grad_(True)

ic_x = 2 * torch.rand(N_ic, 1, device=device) - 1
ic_t = torch.zeros(N_ic, 1, device=device)
ic = torch.cat([ic_x, ic_t], dim=1)
u_ic_true = -torch.sin(np.pi * ic_x)

# Boundary conditions: u(−1, t) = u(1, t) = 0
bc_t     = torch.rand(N_bc, 1, device=device)
bc_left  = torch.cat([-torch.ones(N_bc, 1, device=device), bc_t], dim=1)
bc_right = torch.cat([ torch.ones(N_bc, 1, device=device), bc_t], dim=1)
bc = torch.cat([bc_left, bc_right], dim=0)

### 2.4 Training

We use four weighted loss terms:

$$\mathcal{L} = \mathcal{L}_{\text{PDE}} + 10\,\mathcal{L}_{\text{IC}} + \mathcal{L}_{\text{BC}} + 50\,\mathcal{L}_{\text{data}}$$

- **IC weight = 10**: the initial condition is the primary driver of the solution shape; stronger enforcement helps the network settle into the correct regime early in training.
- **Data weight = 50**: the data loss is averaged over 1,000 noisy points while the PDE residual is averaged over 20,000 collocation points — the higher weight compensates for the smaller per-point signal. The systematic way to set these is shown in Section 3.

In [ ]:
model = InversePINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20000
nu_history = []

for epoch in range(epochs):
    optimizer.zero_grad()

    # PDE residual using learnable model.nu
    u = model(colloc)
    grads = torch.autograd.grad(
        u, colloc,
        grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]
    u_x = grads[:, 0:1]
    u_t = grads[:, 1:2]
    u_xx = torch.autograd.grad(
        u_x, colloc,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]
    residual = u_t + u * u_x - model.nu * u_xx
    loss_pde = torch.mean(residual**2)

    # IC loss
    loss_ic = torch.mean((model(ic) - u_ic_true)**2)

    # BC loss: u(±1, t) = 0
    loss_bc = torch.mean(model(bc)**2)

    # Data loss
    loss_data = torch.mean((model(data_xt) - u_noisy)**2)

    loss = loss_pde + 10 * loss_ic + loss_bc + 50 * loss_data
    loss.backward()
    optimizer.step()

    nu_history.append(model.nu.item())
    if epoch % 4000 == 0:
        print(f"Epoch {epoch:6d} | Loss: {loss.item():.3e} | "
              f"PDE: {loss_pde.item():.3e} | IC: {loss_ic.item():.3e} | "
              f"BC: {loss_bc.item():.3e} | Data: {loss_data.item():.3e} | "
              f"nu: {model.nu.item():.6f}")

print(f"\nTrue nu:     {true_nu:.6f}")
print(f"Inferred nu: {model.nu.item():.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(nu_history, lw=1.5, label='Inferred $\\nu$')
ax.axhline(true_nu, color='r', ls='--', lw=2, label=f'True $\\nu = {true_nu:.5f}$')
ax.set_xlabel('Epoch')
ax.set_ylabel('$\\nu$')
ax.set_title('Convergence of Inferred Viscosity')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Loss weighting strategies

Multi-term losses often compete. If one term is orders of magnitude larger than others, the optimiser ignores the smaller ones. Common strategies:

| Strategy | Description | Pros / Cons |
|---|---|---|
| **Manual tuning** | Fixed weights chosen by inspection | Simple, but tedious |
| **Normalisation** | Divide each loss by its initial value | Keeps losses on similar scale |
| **NTK balancing** | Weights based on Neural Tangent Kernel (NTK) eigenvalues | Principled, but expensive |
| **Gradient normalisation** | Set $\lambda_i \propto 1 / \|\nabla_{\theta} \mathcal{L}_i\|$ | Effective, moderate cost |
| **Self-adaptive weights** | Make $\lambda_i$ learnable (with constraints) | Flexible, risk of degenerate solutions |

A practical rule of thumb: start with equal weights, observe which loss term is slowest to decrease, and increase its weight. Repeat until all components converge together.

### 3.1 Gradient-based weight normalisation

In [ ]:
def compute_grad_norm(loss, model):
    """L2 norm of gradients of loss w.r.t. all model parameters."""
    grads = torch.autograd.grad(loss, model.parameters(),
                                retain_graph=True, allow_unused=True)
    return sum(g.norm()**2 for g in grads if g is not None).sqrt().item()


# ── Compute initial gradient norms on a freshly initialised model ─────────────
torch.manual_seed(99)
demo_model = InversePINN().to(device)

demo_col_x = 2 * torch.rand(2000, 1, device=device) - 1
demo_col_t = torch.rand(2000, 1, device=device)
demo_col   = torch.cat([demo_col_x, demo_col_t], dim=1).requires_grad_(True)

u_c    = demo_model(demo_col)
g_c    = torch.autograd.grad(u_c, demo_col, torch.ones_like(u_c),
                              create_graph=True, retain_graph=True)[0]
u_x_c  = g_c[:, 0:1]
u_t_c  = g_c[:, 1:2]
u_xx_c = torch.autograd.grad(u_x_c, demo_col, torch.ones_like(u_x_c),
                              create_graph=True, retain_graph=True)[0][:, 0:1]
l0_pde  = torch.mean((u_t_c + u_c * u_x_c - demo_model.nu * u_xx_c) ** 2)

# IC and data losses are fresh forward passes — independent computation graphs
l0_ic   = torch.mean((demo_model(ic)      - u_ic_true) ** 2)
l0_data = torch.mean((demo_model(data_xt) - u_noisy)   ** 2)

gn_pde  = compute_grad_norm(l0_pde,  demo_model)
gn_ic   = compute_grad_norm(l0_ic,   demo_model)
gn_data = compute_grad_norm(l0_data, demo_model)

mean_gn = (gn_pde + gn_ic + gn_data) / 3
w_pde   = mean_gn / gn_pde  if gn_pde  > 0 else 1.0
w_ic    = mean_gn / gn_ic   if gn_ic   > 0 else 1.0
w_data  = mean_gn / gn_data if gn_data > 0 else 1.0

print("Gradient norms at random initialisation:")
print(f"  ||∇L_pde||   = {gn_pde:.4f}")
print(f"  ||∇L_ic||    = {gn_ic:.4f}")
print(f"  ||∇L_data||  = {gn_data:.4f}")
print()
print("Gradient-norm balanced weights (λ_i ∝ mean_norm / norm_i):")
print(f"  w_pde   = {w_pde:.2f}")
print(f"  w_ic    = {w_ic:.2f}")
print(f"  w_data  = {w_data:.2f}")
print()
print("We use fixed weights λ_ic=10, λ_data=50 — a manual approximation.")
print("A production implementation would recompute these norms every N epochs.")

## 4. 2D Poisson equation

We now extend PINNs to an elliptic PDE in 2D space (no time variable):

$$
\nabla^2 u = f(x, y), \quad (x, y) \in [0, 1]^2
$$

$$
u = 0 \text{ on } \partial\Omega
$$

We choose $f = -2\pi^2 \sin(\pi x)\sin(\pi y)$, which gives the exact solution $u = \sin(\pi x)\sin(\pi y)$.

The PINN residual is:
$$
r = \frac{\partial^2 u_\theta}{\partial x^2} + \frac{\partial^2 u_\theta}{\partial y^2} - f(x, y)
$$

The network takes 2D input $(x, y)$; there is no time dimension. Boundary conditions are enforced on all four sides of the unit square.

### 4.1 Model and sampling

In [ ]:
class PoissonPINN(nn.Module):
    """MLP: input (x, y), output u. No time."""
    def __init__(self, layers=[2, 50, 50, 50, 1]):
        super().__init__()
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i+1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)

    def forward(self, X):
        return self.net(X)


N_col_2d = 8000
N_bc_2d  = 200
torch.manual_seed(7)

# Interior collocation
col2d = torch.rand(N_col_2d, 2, device=device).requires_grad_(True)

# Boundary: all four sides of [0,1]^2.  s_bc is a spatial parameter, not time.
s_bc   = torch.rand(N_bc_2d, device=device)
zeros  = torch.zeros(N_bc_2d, device=device)
ones   = torch.ones(N_bc_2d, device=device)
bc_bottom = torch.stack([s_bc, zeros], dim=1)
bc_top    = torch.stack([s_bc, ones],  dim=1)
bc_left   = torch.stack([zeros, s_bc], dim=1)
bc_right  = torch.stack([ones,  s_bc], dim=1)
bc2d = torch.cat([bc_bottom, bc_top, bc_left, bc_right], dim=0)

### 4.2 Training

In [ ]:
poisson_model = PoissonPINN().to(device)
poisson_opt = torch.optim.Adam(poisson_model.parameters(), lr=1e-3)

poisson_history = {'pde': [], 'bc': []}

for epoch in range(8000):
    poisson_opt.zero_grad()

    u2d = poisson_model(col2d)
    grads = torch.autograd.grad(
        u2d, col2d,
        grad_outputs=torch.ones_like(u2d),
        create_graph=True, retain_graph=True
    )[0]
    # ∂²u/∂x²: differentiate ∂u/∂x (col 0) w.r.t. col2d, keep col 0
    u_x2 = torch.autograd.grad(
        grads[:, 0:1], col2d,
        grad_outputs=torch.ones_like(grads[:, 0:1]),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]
    # ∂²u/∂y²: differentiate ∂u/∂y (col 1) w.r.t. col2d, keep col 1
    u_y2 = torch.autograd.grad(
        grads[:, 1:2], col2d,
        grad_outputs=torch.ones_like(grads[:, 1:2]),
        create_graph=True, retain_graph=True
    )[0][:, 1:2]

    f_val = -2 * np.pi**2 * torch.sin(np.pi * col2d[:, 0:1]) * torch.sin(np.pi * col2d[:, 1:2])
    residual_2d = u_x2 + u_y2 - f_val
    loss_pde_2d = torch.mean(residual_2d**2)

    loss_bc_2d = torch.mean(poisson_model(bc2d)**2)

    loss_2d = loss_pde_2d + loss_bc_2d
    loss_2d.backward()
    poisson_opt.step()

    poisson_history['pde'].append(loss_pde_2d.item())
    poisson_history['bc'].append(loss_bc_2d.item())

    if epoch % 2000 == 0:
        print(f"Epoch {epoch:6d} | PDE: {loss_pde_2d.item():.3e} | BC: {loss_bc_2d.item():.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(poisson_history['pde'], label='PDE residual')
ax.semilogy(poisson_history['bc'],  label='BC loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('2D Poisson — Training Loss')
ax.legend()
plt.tight_layout()
plt.show()

### 4.3 Results

In [ ]:
with torch.no_grad():
    Ng = 80
    xi = torch.linspace(0, 1, Ng)
    yi = torch.linspace(0, 1, Ng)
    Xi, Yi = torch.meshgrid(xi, yi, indexing='ij')
    XY = torch.stack([Xi.flatten(), Yi.flatten()], dim=1).to(device)
    u_pinn_2d = poisson_model(XY).cpu().numpy().reshape(Ng, Ng)

u_exact_2d = np.sin(np.pi * Xi.numpy()) * np.sin(np.pi * Yi.numpy())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].contourf(Xi.numpy(), Yi.numpy(), u_pinn_2d, levels=40, cmap='viridis')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('PINN Solution')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

im1 = axes[1].contourf(Xi.numpy(), Yi.numpy(), u_exact_2d, levels=40, cmap='viridis')
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Exact Solution')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

error_2d = np.abs(u_pinn_2d - u_exact_2d)
im2 = axes[2].contourf(Xi.numpy(), Yi.numpy(), error_2d, levels=40, cmap='hot_r')
plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Absolute Error')
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')

plt.suptitle('PINN for 2D Poisson Equation', fontsize=13)
plt.tight_layout()
plt.show()

rel_l2_2d = np.sqrt(np.mean(error_2d**2)) / np.sqrt(np.mean(u_exact_2d**2))
print(f'Relative L2 error: {rel_l2_2d:.4e}')

## 5. Summary

This session covered three practical extensions of the basic PINN framework:

1. **Inverse problems**: make unknown parameters `nn.Parameter` objects, add a data loss term, and train jointly. Use log-parameterisation for strictly positive quantities.
2. **Loss weighting**: balance terms manually (inspect loss magnitudes), by gradient normalisation, or via adaptive schemes. This is often the most important practical tuning knob.
3. **2D Poisson**: the extension to 2D is straightforward — the network takes 2D input, boundary conditions cover all four sides, and the Laplacian requires two second-derivative computations.

**Assignment**: implement the inverse Burgers' experiment with different noise levels (0%, 1%, 5%) and report how the inferred $\nu$ changes. Bonus: implement gradient-norm-based loss weighting and compare convergence.

**Reading**: Wang, Yu & Perdikaris (2022) *When and why PINNs fail to train: A neural tangent kernel perspective*, Journal of Computational Physics. This gives the theoretical foundation for NTK-based loss balancing.

**[Session 9](Session9.ipynb)** takes a step back from complex PDEs to study a single tuning decision that affects every PINN: choosing the physics loss weight $\lambda$. Using the simple harmonic oscillator as a benchmark, we sweep $\lambda$ systematically, measure interpolation and extrapolation error, and derive a programmatic procedure for finding $\lambda^*$.

## Exercises

1. **Noise sensitivity**: retrain the inverse Burgers' PINN with noise levels $\sigma \in \{0, 0.01, 0.05, 0.1\}$. Plot the inferred $\nu$ vs $\sigma$ and report whether the data weight ($\lambda_{\text{data}} = 50$) requires adjustment as noise increases.

2. **Alternative Poisson source**: solve the 2D Poisson equation with $f(x, y) = -5\pi^2 \sin(\pi x)\sin(2\pi y)$, for which the exact solution is $u = \sin(\pi x)\sin(2\pi y)$. Adapt the network and training from Section 4. Compare the relative $L_2$ error with the $\sin(\pi x)\sin(\pi y)$ case — is this problem harder or easier, and why?

3. **Gradient-norm weighting**: use the `compute_grad_norm` function from Section 3 to automatically set the IC and data weights ($\lambda_{\text{IC}}$ and $\lambda_{\text{data}}$) for the inverse Burgers' problem. Retrain with these computed weights and compare the convergence of the inferred $\nu$ history with the manually chosen $\lambda_{\text{IC}} = 10$, $\lambda_{\text{data}} = 50$.

4. **Initial guess sensitivity**: train the inverse Burgers' PINN with `log_nu` initialised to $\log(10\nu_{\text{true}})$ (ten times the true value) and to $\log(0.1\nu_{\text{true}})$ (one tenth). Does the model always converge to the correct $\nu$? What does this suggest about the convexity of the loss landscape with respect to the learnable parameter?